# Conversational memory agent — interactive tour

A LangGraph ReAct agent (`langchain.agents.create_agent`) that answers from the
news corpus via a retriever tool, with its conversation state persisted in
AgensGraph by **`AgensSaver`** (the LangGraph checkpointer). The same `thread_id`
resumes the conversation — even from a brand-new agent instance. The transcript
is also mirrored to **`AgensChatMessageHistory`**.

**Prerequisites:** the news store (demo 03's `ingest.py`) and an `OPENAI_API_KEY`.

In [1]:
import asyncio
import sys, pathlib

HERE = pathlib.Path.cwd()                  # .../04_chat_memory_agent
p = HERE
while p != p.parent and not (p / "_common").is_dir():
    p = p.parent
sys.path.insert(0, str(p))                 # demos root, for _common
sys.path.insert(0, str(HERE))              # this dir, for `agent`

import agent                               # build_agent / ask / search_news tool live here
from _common import agens
from langchain_core.messages import AIMessage, HumanMessage
from langchain_agensgraph import AgensChatMessageHistory

THREAD = "notebook-demo"
bot, saver = agent.build_agent()           # ReAct agent + AgensSaver checkpointer
saver.delete_thread(THREAD)                # start this demo thread clean
print("agent ready — AgensSaver checkpointing graph 'agent_memory'")

agent ready — AgensSaver checkpointing graph 'agent_memory'


## A multi-turn conversation

Each turn is persisted by the checkpointer under `THREAD`; later turns see the
earlier ones (and the tool results) without re-sending them.

In [2]:
print(agent.ask(bot, THREAD,
    "Search the news for stories about artificial intelligence and summarize the main themes."))

The recent news stories about artificial intelligence (AI) highlight several key themes:

1. **Applications of AI**: AI is being utilized in various sectors, including customer service, where it helps route inquiries to the right agents and assists with responses. Autonomous vehicles are also mentioned as a significant application of machine learning.

2. **Job Automation vs. Augmentation**: There is an ongoing discussion about the impact of AI on the job market. While some fear job losses due to automation, others explore the concept of augmentation, where AI enhances human capabilities rather than replacing them.

3. **AI in Human Resources**: Major companies are increasingly using AI for managing human resources, including hiring processes. Predictions suggest significant growth in the use of AI in this area, indicating a shift in how organizations approach workforce management.

4. **Concerns about Artificial General Intelligence**: The potential development of artificial general i

In [3]:
print(agent.ask(bot, THREAD, "Which of those themes relates most to jobs or hiring?"))

The themes that relate most to jobs or hiring are:

1. **Job Automation vs. Augmentation**: This theme addresses the concern that AI could lead to job losses due to automation. However, it also explores the idea of augmentation, where AI tools enhance human work rather than replace it, suggesting a potential for new job roles that leverage AI.

2. **AI in Human Resources**: This theme specifically focuses on how companies are using AI in hiring processes and workforce management. It indicates a trend towards integrating AI technologies to improve efficiency and decision-making in HR practices, which directly impacts job recruitment and employee management.

Both themes highlight the evolving relationship between AI and the job market, emphasizing both the challenges and opportunities presented by AI in the context of employment.


In [4]:
print(agent.ask(bot, THREAD, "Give one concrete example from the articles you found."))

One concrete example from the articles is the use of AI in hiring processes by major companies. The articles mention that organizations are leveraging AI technologies to manage human resources, which includes automating aspects of recruitment. For instance, AI can analyze resumes, screen candidates, and even assist in scheduling interviews, thereby streamlining the hiring process and improving efficiency. This trend reflects a significant shift in how companies approach workforce management, with predictions of substantial growth in the adoption of AI tools in HR practices.


## Resume from a checkpoint — a brand-new agent instance

`build_agent()` constructs a fresh agent + a fresh `AgensSaver` (as a new process
would). Using the same `thread_id`, it picks up the full prior state from
AgensGraph — so it can answer about earlier turns without searching again.

In [5]:
bot2, _ = agent.build_agent()             # fresh instance, same persisted thread
print(agent.ask(bot2, THREAD,
    "Without searching again, what was my very first question in this conversation?"))

Your very first question in this conversation was to search the news for stories about artificial intelligence and summarize the main themes.


## Long-term memory (`AgensStore`) — not scoped to a conversation

The checkpointer above resumes *one* thread. `AgensStore` holds what should outlive any of
them: it is keyed by who a fact is about, not by which conversation mentioned it. The agent
reaches it through the `remember_about_user` / `recall_about_user` tools.


In [6]:
OTHER = "notebook-other-thread"

print(agent.ask(bot, THREAD, "Remember that I work on graph databases and prefer short answers."))
print()
# A different thread_id: the checkpointer knows nothing of the exchange above, so anything
# recalled here came from the store rather than from the conversation.
print(agent.ask(bot, OTHER, "What do you already know about me? Do not search the news."))


Got it! I will remember that you work on graph databases and prefer short answers.



I know that you work on graph databases and prefer short answers.


### Namespaces are a hierarchy

An item records every namespace that contains it, so searching a parent finds everything
beneath it. That is an equality inside a list rather than a comparison between two strings,
which matters because a namespace is stored as jsonb and jsonb compares under the database's
collation — a range over the path text returns the wrong rows on a linguistic one.


In [7]:
facts = agent.facts_store()

for item in facts.search(("memories", agent.USER), limit=10):
    print(f"  {item.namespace}  {item.key[:8]}  {item.value['text']}")

print("\nsearching the parent namespace ('memories',) finds every user's items:")
for item in facts.search(("memories",), limit=10):
    print(f"  {item.namespace}  {item.value['text']}")

print("\nnamespaces in use:", facts.list_namespaces())


  ('memories', 'demo-user')  4e09787a  User works on graph databases.
  ('memories', 'demo-user')  d081b957  User prefers short answers.

searching the parent namespace ('memories',) finds every user's items:
  ('memories', 'demo-user')  User works on graph databases.
  ('memories', 'demo-user')  User prefers short answers.

namespaces in use: [('memories', 'demo-user')]


### Writing many memories at once

An agent learns one thing at a time, which is what `put` is for. Seeding or importing is
different: every `put` embeds its own text, so a loop of them is one model round trip per
memory. `batch` embeds the whole set in a single request.


In [8]:
import time
from langgraph.store.base import PutOp

SEED = ("memories", "seeded-user")
facts_to_load = [
    "prefers metric units", "works in the Asia/Seoul timezone",
    "reads documentation before asking", "dislikes long preambles",
    "uses Python and SQL daily", "runs AgensGraph locally",
    "prefers tables to prose", "asks for sources",
]

def clear():
    for it in facts.search(SEED, limit=100):
        facts.delete(SEED, it.key)

clear()
started = time.perf_counter()
for i, text in enumerate(facts_to_load):
    facts.put(SEED, f"loop{i}", {"text": text})
one_at_a_time = time.perf_counter() - started

clear()
started = time.perf_counter()
facts.batch([PutOp(namespace=SEED, key=f"bat{i}", value={"text": t})
             for i, t in enumerate(facts_to_load)])
batched = time.perf_counter() - started

n = len(facts_to_load)
print(f"{n} memories, one put() each : {one_at_a_time:5.2f}s  ({n/one_at_a_time:4.1f}/s)")
print(f"{n} memories, one batch()    : {batched:5.2f}s  ({n/batched:4.1f}/s)   "
      f"{one_at_a_time/batched:.0f}x")
print(f"\nstored: {len(facts.search(SEED, limit=100))} memories")
clear()


8 memories, one put() each :  2.94s  ( 2.7/s)
8 memories, one batch()    :  0.83s  ( 9.7/s)   4x

stored: 8 memories


## Managing a thread — fork, trim, drop a run

A thread gains a checkpoint per step, which is what makes it resumable and what makes it
grow. `copy_thread` forks one, `prune` trims the history a resumable thread no longer
needs, and `delete_for_runs` removes the checkpoints of a particular invocation across
whatever threads it touched.


In [9]:
FORK = f"{THREAD}-forked"

saver.delete_thread(FORK)          # copying onto an occupied thread is refused
saver.copy_thread(THREAD, FORK)
print(f"{THREAD}: {agent.checkpoint_count(saver, THREAD)} checkpoints")
print(f"{FORK}: {agent.checkpoint_count(saver, FORK)} (a copy — independent from here)")

# the fork carries the whole parent chain, so it resumes on its own
print("\n" + agent.ask(bot, FORK, "In one sentence, what have we been discussing?"))
print(f"\nonly the fork grew: {THREAD}={agent.checkpoint_count(saver, THREAD)}, "
      f"{FORK}={agent.checkpoint_count(saver, FORK)}")


notebook-demo: 19 checkpoints
notebook-demo-forked: 19 (a copy — independent from here)



We have been discussing recent news themes related to artificial intelligence, particularly its impact on jobs and hiring.

only the fork grew: notebook-demo=19, notebook-demo-forked=22


In [10]:
saver.prune([FORK])
current = saver.get_tuple({"configurable": {"thread_id": FORK}})
print(f"after prune, {FORK}: {agent.checkpoint_count(saver, FORK)} checkpoints, "
      f"still resumable: {current is not None}")

# what prune kept per channel: a channel recording deltas is rebuilt from these
history = saver.get_delta_channel_history(
    config=current.config, channels=sorted(current.checkpoint["channel_values"]))
for channel, entry in sorted(history.items()):
    print(f"  {channel}: {len(entry.get('writes', []))} write(s), "
          f"seed {'kept' if 'seed' in entry else 'not needed'}")

saver.delete_thread(FORK)


after prune, notebook-demo-forked: 3 checkpoints, still resumable: True
  __pregel_tasks: 0 write(s), seed kept
  messages: 1 write(s), seed kept


## Concurrent conversations — the awaiting path

`AsyncAgensSaver` is `AgensSaver`; the same object answers both surfaces. What awaiting
buys is that separate conversations overlap: each one's checkpoint reads and writes take
a connection from the pool while the others wait on the model.

One loop for the whole thing — a pool's workers are tasks of the loop that opened it, so
an engine used across two of them has to rebuild it for the second. The kernel already
runs one, so this cell awaits directly where `agent.py` calls `asyncio.run`.


In [11]:
import time

async def answer_all(pairs):
    answers = await asyncio.gather(*(
        bot.ainvoke({"messages": [{"role": "user", "content": text}]},
                    config={"configurable": {"thread_id": t}})
        for t, text in pairs))
    resumable = [(await saver.aget_tuple({"configurable": {"thread_id": t}})) is not None
                 for t, _ in pairs]
    return answers, resumable

pairs = [("nb-async-a", "In one word, name a technology in the news."),
         ("nb-async-b", "In one word, name a place in the news."),
         ("nb-async-c", "In one word, name a company in the news.")]
for t, _ in pairs:
    saver.delete_thread(t)

# `await` rather than `asyncio.run`: the notebook kernel is already running a loop.
started = time.perf_counter()
answers, resumable = await answer_all(pairs)
elapsed = time.perf_counter() - started

for (t, _), out in zip(pairs, answers):
    print(f"  [{t}] {out['messages'][-1].content.strip()[:60]}")
print(f"\n{len(pairs)} conversations in {elapsed:.1f}s; each resumable: {resumable}")

for t, _ in pairs:
    saver.delete_thread(t)


  [nb-async-a] "Robotics"
  [nb-async-b] Baton Rouge
  [nb-async-c] Hyatt

3 conversations in 2.4s; each resumable: [True, True, True]


## `AgensChatMessageHistory` — a simple per-session message log

A lighter-weight memory primitive: append/read messages keyed by a session id.

In [12]:
history = AgensChatMessageHistory(
    THREAD, graph=agens.make_graph("chat_log", create=True, refresh_schema=False))
history.clear()
history.add_messages([
    HumanMessage(content="What are the main AI themes in the news?"),
    AIMessage(content="Applications, job automation/augmentation, corporate adoption, AGI concerns."),
])
print(f"{len(history.messages)} messages for session {THREAD!r}:")
for m in history.messages:
    print(f"  {m.type:9} {m.content[:60]}")

2 messages for session 'notebook-demo':
  human     What are the main AI themes in the news?
  ai        Applications, job automation/augmentation, corporate adoptio


## What you can do with this

- **Durable agents**: `AgensSaver` persists full LangGraph state per `thread_id`,
  so conversations survive process restarts and resume exactly where they left off.
- **Memory that outlives a thread**: `AgensStore` keys a fact by who it is about, so
  one conversation can learn it and the next can recall it. Namespaces nest, and a
  search of a parent finds everything beneath it.
- **Grounded tools**: the agent answers from the AgensgraphVector news store
  (demo 03) via a retriever tool — graph, vectors, and agent memory in one DB.
- **Chat history**: `AgensChatMessageHistory` is a drop-in per-session message log.

Run one turn at a time from the shell to see true cross-process resume:

```bash
.venv/bin/python examples/demos/04_chat_memory_agent/agent.py my-thread "your message"
```

Close the shared pool when done: `agens.close()`